### POC notebook where everything is written in raw python (no function wrapping), for tutorial purpose only 

In [ ]:
from dotenv import load_dotenv
load_dotenv() # if it says true then it means that it's successfully loaded the .env file

True

In [4]:
from openai import OpenAI
openai_client = OpenAI()

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
llm("capital of cabo verde")

'The capital of Cabo Verde is **Praia**.'

In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—if the course is still open, you can usually join after it has started. A few things to check:

- **Enrollment deadline:** Some courses allow late joining only for a limited time.
- **Prerequisites:** Make sure you meet any required background or qualifications.
- **Access to past materials:** Ask whether you’ll get access to previous lectures, assignments, and recordings.
- **Catch-up plan:** See if there’s a way to make up missed work.

If you want, I can help you draft a short message to the course instructor asking about late enrollment.


In [9]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [10]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [35]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [36]:
# fetch all docs

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status() # if something is broken, exit
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)


1350

In [13]:
documents[500]

{'id': 'b5d37f88b6',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How are homework assignments scored?',
 'answer': "Homework grade = points for the questions + 1 point for the FAQ + up to 7 points for learning in public.\n\n- Learning in public: up to 7 points, depending on how many platforms you shared your post on (e.g. LinkedIn, X, blog) and the quality of the post.\n- FAQ: 1 point. To get it, contribute to the FAQ repo and add the link to your PR in your homework submission.\n\nOptional questions are scored too - read the submission form carefully. The homework is for practice and does not affect your certificate. If you think you weren't graded, log in to check your results or search for your name on the leaderboard."}

# Search 

Indexing with minsearch

In [14]:
from minsearch import Index

In [15]:
index = Index(
    text_fields =['question','section','answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [22]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5} #if a field is more important to search, then boost its relevancy in search results
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
search_results = index.search(question,
             filter_dict = {"course":"llm-zoomcamp"},
             boost_dict = {'question':2.0, 'section':1.0, 'answer': 1.0}, 
             num_results=5)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## Prompt

In [24]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [25]:
USER_PROMPT_TEMPLATE = '''
Question: 
{question}

Context:
{context}
'''

In [26]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append("")
    
    return '\n'.join(lines).strip()

In [31]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
        )
    return prompt.strip()

In [32]:
prompt = build_prompt(question,search_results)

In [41]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [43]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0fed03901e5d5177006a3cebe0797c81a1a4faec3b53e632f9",
  "created_at": 1782377440.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0fed03901e5d5177006a3cebe13c7881a1966759a944ae5b66",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can join now.\n\nYou can start learning and submitting homework as long as the submission form is still open. If you want to get a certificate, make sure you submit your project while the course is still accepting submissions.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed

In [ ]:
response.usage # shows the # of input, cached & output tokens, 

ResponseUsage(input_tokens=481, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=50, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=531)

# Estimate cost

In [45]:
input_price = 0.75 / 1_000_000 # for gpt-5.4 mini
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00058575

# Add memory

In [49]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [50]:
print(llm(INSTRUCTIONS,prompt))

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.


## Full RAG

In [52]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [53]:
answer = rag(question)
print(answer)

Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.
